In [ ]:
# =============================
# Imports, seeds
# =============================
import os
import sys
import math
import time
import random
import warnings

import h5py
import pywt
import numpy as np
import pandas as pd
import matplotlib
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from skimage.metrics import structural_similarity as ssim
from scipy import stats

# Headless execution (smoke tests, nbconvert): matplotlib otherwise defaults to
# tkagg on Windows and tries to open GUI windows. Jupyter runs are unaffected.
if os.environ.get("FINAL_LARA_HEADLESS") == "1":
    matplotlib.use("Agg")

import matplotlib.pyplot as plt

try:
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")
except (AttributeError, OSError):
    pass
warnings.filterwarnings("ignore")

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)


# Final LARA — Seismic Autoencoder Compression Study

Notebook `Final_LARA_27092026`, merged from `Final_HARA_20092026` and `LARA_27092026`.

`Final_HARA_20092026` is the skeleton: the Wavelet baselines, the nine plotting
helpers, the multi-model comparison loop, the significance test and the efficiency
analysis. `LARA_27092026` contributes the `LARA` class, the per-sample
`evaluate_model`, and the locked configuration / training recipe.

**`HeavyAdvancedResidualAutoencoder` is deleted, not kept. `LARA` occupies its slot
in the comparison**, so the model list is now
`LARA`, `GeneralizedAutoencoder`, `AE_PureConcat`, `Wavelet_DB4`, `Wavelet_SYM8`,
`Wavelet_COIF3` across the same CR grid `[2, 3, 5, 10, 15, 20, 30, 50, 60, 100]`.

## LARA architecture (`variant="pyramid_funnel32"`)

| Component | Setting |
|---|---|
| Encoder | `stage0` Conv1d(1->32, k15, s2) + BN + ELU + `AttentionBlock` -> `EnhancedResidualBlock`(32->64, s2, attn) -> `EnhancedResidualBlock`(64->128, s2, attn) -> (+ `EnhancedResidualBlock`(128->256, s2, attn) for cr>30) |
| Bottleneck | `bot_conv` 1x1 conv to `BOTTLE_CHANNELS=32` (cr>=10; 16 for cr<10) + ELU, then `Linear(BOTTLE_CHANNELS*T -> latent)` with **`T = L`** (the *filled funnel*: no `min(latent, L)` clamp) |
| Latent | exactly `1500 // cr`, asserted in `test_dimensions` |
| Decoder | `Linear(latent -> BOTTLE_CHANNELS*T)` + `Unflatten` + ELU + 6-level channel-pyramid learned upsampler + refinement head (Conv -> ELU -> Conv -> Sigmoid) |

Parameter counts measured on this machine: 4.73 M at CR=2 down to 0.68 M at CR=100.
`LARA` has no `count_params()` method (the removed Heavy/Light classes did);
parameter counts come from `evaluate_model`, which sums `p.numel()` directly.

## What changed relative to each parent

**From `Final_HARA_20092026`:**
- `HeavyAdvancedResidualAutoencoder` removed; `LARA` replaces it in the model list.
- `train_model_heavy_ablation` -> `train_model_main`, now parameterised by `RECIPES`
  so the training budget is a config value rather than hardcoded.
- `load_filtered_z_waveforms` -> `load_filtered_stead(csv, h5, max_samples, component)`.
  The broken `r"C:\\Users\\hafiz\\..."` path literal is gone; `CSV_PATH` / `H5_PATH`
  and `COMPONENT` are config values.
- Checkpoints moved from the working directory into `CKPT_DIR`; `best_file()` is the
  single accessor used for save, load and cleanup.
- `evaluate_model` now reports per-sample statistics (std, median, P25, P75, `GradL1`)
  instead of only whole-set aggregates, and keeps the Wavelet branch so the three
  Wavelet baselines still evaluate.
- `statistical_significance_test` skips model pairs with fewer than 2 CRs instead of
  emitting a divide-by-zero `nan`.

**From `LARA_27092026`:**
- The single-model study loop is replaced by the 6-model comparison loop.
- The run log (`log()`, `run_log_*.txt`), the spectral loss classes
  (`SpectralTransientLoss`, `EnhancedSpectralTransientLoss`) and the unused
  `LightCapacityResidualAutoencoder` were **not** carried over.
- The `VARIANT` knob is gone; `pyramid_funnel32` is used directly so the architecture
  is provably the validated one. The `variant` argument stays on the class.
- `MAX_EPOCHS` / `PATIENCE` moved into the `RECIPES` dict.
- **The training pool grew from 14,983 to 25,091 traces** (`MIN_MAGNITUDE` 3.0 -> 2.5).
  See "The training pool changed" above — this is the one change that makes the numbers
  non-comparable with the parent notebook's results.

## Training recipe

`LARA_RECIPE` selects the hyper-parameter set. The default is `"heavy"`, which
reproduces exactly what `Final_HARA_20092026` used for `HeavyAdvancedResidual`
(Adam `lr=2e-4`, `weight_decay=1e-5`, 120-epoch cap, `ReduceLROnPlateau` patience 5,
early-stopping patience 8, grad clip 1.0, `nn.MSELoss`). Keeping the default means LARA
is directly comparable with the other models *in this same run*.

Set `LARA_RECIPE = "study"` to use LARA_27092026's own recipe instead
(Adam `lr=5e-4`, no weight decay, 200-epoch cap). That is the configuration LARA was
validated under, but its numbers are then no longer comparable to the other models
trained in the same run. Both recipes use plain `nn.MSELoss()`.

## Configuration

| Setting | Value |
|---|---|
| Data | STEAD filtered pool, all traces (**~25,091**): `trace_category == earthquake_local`, `source_magnitude > 2.5`, `source_distance_km <= 60` |
| Channel | `COMPONENT = 2` (Z) |
| Split | 80 / 10 / 10 → **20,073 / 2,509 / 2,509**, `SEED = 42` |
| Normalisation | per-trace min-max inside `EfficientSeismicDataset` |
| Evaluation | `N_EVAL = 200` traces from the held-out test split (`None` = all) |
| CRs | `[2, 3, 5, 10, 15, 20, 30, 50, 60, 100]` |
| Batch / workers | `BATCH = 16`, `WORKERS = 2` |
| Checkpoint | best-by-validation-MSE -> `ckpts\best_{model}_cr{CR}.pth` |
| Outputs | `results\model_comparison_results.csv`, `model_comparison_summary.csv`, figures |

The filter thresholds are `MIN_MAGNITUDE` and `MAX_DISTANCE_KM` in the CONFIG cell, not
hardcoded in the loader.

Wavelet baselines are non-learned transforms, so their `Parameter_Count` and
`Model_Size_MB` are reported as `0`, and they are excluded from the three
parameter plots (`plot_parameter_comparison`, `plot_parameter_vs_performance`,
`plot_comprehensive_training_analysis`) to keep a log axis free of zeros. They do
appear in all the performance plots.

## Known quirks carried over from the parents (not bugs introduced here)

- `AE_PureConcat`'s parameter count is **non-monotonic at CR=30** (0.68 M, below the
  0.77 M at CR=50) because `model_2_architecture` uses 8 feature maps at CR=30 and 16
  everywhere else. Don't read the parameter plots as monotone for that model.
- `statistical_significance_test` compares the per-CR summary values, not per-trace
  values, so its p-values are weak evidence.
- `LARA`'s parameter count ticks *up* slightly from CR=5 (2.03 M) to CR=10 (2.05 M):
  that is the funnel filling in (`BOTTLE_CHANNELS` 16 -> 32 with `T = L`), which is the
  intended LARA design.

## The training pool changed: 14,983 -> 25,091 traces

`MIN_MAGNITUDE` was lowered from **3.0 to 2.5** (`MAX_DISTANCE_KM` stays 60), so the
study pool grows from 14,983 to **25,091** traces and the train split from 11,986 to
**20,073**.

> **This means results from this notebook are NOT directly comparable to the
> `LARA_27092026` run**, which used the 14,983-trace pool, nor to the older 2,000-trace
> reference figures quoted in that notebook's docs. Different training data, so do not
> read the two sets of numbers as a like-for-like comparison.

The change was made deliberately, on measurements of the STEAD picks rather than on
count alone. LARA only ever sees the **first 1500 samples (15 s)** of each 6000-sample
trace, so the filter is bounded by what is inside that window. Measured on this STEAD
build (1,030,231 local events, all with picked P and S):

| Distance band | Traces in pool | **S inside the 15 s window** | post-P / pre-P energy |
|---|---|---|---|
| 0–20 km | 3,937 | **95–98 %** | 80.8× |
| 20–40 km | 7,680 | **92 %** | 73.0× |
| 40–60 km | 13,474 | **58–64 %** | 45.9× |
| 60–80 km | 24,211 | **20 %** | 24.8× |
| 80–100 km | 24,385 | **0 %** | 15.5× |
| 100–150 km | 37,508 | **0 %** | 25.7× |

P is inside the window for 100 % of local events (median sample ~700). **S is not** — it
leaves the 15 s window at roughly 45–50 km and is gone by 100 km. Consequences:

- **Distance is the binding constraint, not magnitude.** At `D <= 60`, coverage moves
  only from 88 % (M>1.5) to 75 % (M>3.0), whereas across distance it falls from 98 %
  to 0 %. The 60 km cap sits right at the edge of the cliff.
- `M > 2.5, D <= 60` is a **strict improvement** on the old filter: 67 % more data *and*
  slightly better in-window coverage (76.1 % vs 74.8 %), with all three distance
  sub-bands preserved.
- A larger pool was available and rejected: `M > 2.0, D <= 100` gives 168,859 traces but
  only **53.6 %** of them have the S wave visible, so it mostly adds P-only traces that
  teach less and make the metrics look easier.
- About 24 % of the current pool (≈6,000 traces, the 40–60 km band) is P-plus-coda with
  no S in the window. That is the accepted cost of keeping the 0–60 km range. If you
  would rather have cleaner per-trace signal than range diversity, set
  `MAX_DISTANCE_KM = 45` (≈19k traces, ≈90 % coverage) — but that narrows the range.

`snr_db` is empty in this STEAD build, so the `M 2.0–2.5` band is admitted on
arrival-time evidence alone, not on measured SNR. If you distrust events that small,
`MIN_MAGNITUDE = 3.0` is the conservative fallback.

Memory is not a constraint at this pool size: ~574 MB resident including the two spawned
DataLoader workers, against 31.4 GB available.

## GPU support

The notebook auto-selects the best available accelerator in this order:
**CUDA -> XPU -> MPS -> CPU**. It runs on CPU unchanged if no accelerator is found, so
the same file works on a laptop and on a GPU box with no edits.

Cell 2 prints a **DEVICE REPORT** at startup showing the selected device, the GPU name
and compute capability, the cuDNN version, and — if it falls back to CPU — exactly why.
If your torch is a `+cpu` wheel it will say so and print the CUDA install command.

Switches in the CONFIG cell:

| Setting | Default | Effect |
|---|---|---|
| `DEVICE_OVERRIDE` | `None` | Pin a specific device, e.g. `"cuda"` or `"cuda:1"` |
| `CUDNN_BENCHMARK` | `True` | Autotunes conv algorithms for the fixed `(BATCH, 1, 1500)` input shape |
| `ALLOW_TF32` | `True` | TF32 matmul/conv on Ampere and newer. Faster; small numeric change, applied uniformly to every model so the comparison stays fair |
| `USE_AMP` | `False` | Mixed precision. **Off by default** because it changes numerics; the validated MSE recipe is preserved as-is |
| `PIN_MEMORY` | auto (`True` on CUDA) | Pinned host buffers so CPU->GPU copies are asynchronous |

DataLoader batches are moved with `non_blocking=PIN_MEMORY`, `zero_grad(set_to_none=True)`
is used, and the per-epoch `empty_cache()` is applied for both CUDA and XPU.

Two things to know about the GPU path:

- **Validation always runs in full precision**, even when `USE_AMP=True`, so early
  stopping compares like with like.
- **The best model is reloaded from the checkpoint file** at the end of training rather
  than from an in-memory copy. This is a bug fix: the parent notebooks kept
  `model.state_dict().copy()`, which is a *shallow* copy, so the supposedly-best weights
  were still the live tensors and got overwritten in place by later epochs. The saved
  `.pth` files were always correct; the in-memory reload was not.

**This notebook was developed and smoke-tested on a CPU-only machine**
(torch 2.14.0+cpu, no NVIDIA GPU), so the CUDA path is exercised only by the
auto-detection and the fallback, not by an actual GPU run. The CPU fallback is
smoke-verified end to end.

## How to run

Run all cells in order.

**Smoke test first.** Set `SMOKE = True` in the CONFIG cell (or export
`FINAL_LARA_SMOKE=1`) to run 100 traces at CR=10 for 2 epochs through the entire
pipeline — data, training, evaluation, all figures and the CSV — in about two minutes.
Outputs go to `results_smoke/` and `ckpts_smoke/` so a smoke run can never overwrite
real results. Set `SMOKE = False` for the real run.

**Runtime.** The train split is 20,073 traces, 1.67× the previous 11,986, so costs scale
from the earlier estimates. Worst case (120 epochs, no early stopping) on CPU with 16
torch threads: **LARA ≈ 58 h for all 10 CRs, ≈ 81 h for all three neural networks.**
Early stopping (patience 8) typically cuts this by 2–3×, so expect materially less in
practice. On an NVIDIA GPU, roughly **1.6 h** for the full sweep. Early stopping is doing
a lot of that work, so it is worth watching rather than assuming.


In [ ]:
# =============================
# CONFIG
# =============================
# SMOKE=True shrinks the whole study to 100 traces / 1 CR / 2 epochs so the full
# pipeline (data -> train -> eval -> plots -> CSV) can be verified in ~1-2 minutes.
# Leave it False for the real run.
SMOKE = os.environ.get("FINAL_LARA_SMOKE", "0") == "1"

# ---- device / GPU ----
# The notebook auto-selects the best available accelerator: CUDA -> XPU -> MPS -> CPU.
DEVICE_OVERRIDE = None    # e.g. "cuda", "cuda:1", "xpu", "mps", "cpu". None = auto-detect
ALLOW_TF32       = True   # cuDNN/matmul TF32 on Ampere+: faster, tiny numeric change
CUDNN_BENCHMARK  = True   # autotune conv algorithms for the fixed (BATCH, 1, 1500) shape
USE_AMP          = False  # mixed precision: changes numerics, so off by default
PIN_MEMORY       = None   # None = auto (True on CUDA); pinned host memory speeds transfers

# ---- data ----
CSV_PATH  = r"C:\Users\hafiz\Desktop\Mostafa seismic signals\seismic data\merged.csv"      # STEAD metadata
H5_PATH   = r"C:\Users\hafiz\Desktop\Mostafa seismic signals\seismic data\merged.hdf5"     # STEAD waveforms
N_TRACES  = None                        # None = ALL traces passing the filter (~25,091)
COMPONENT = 2                           # STEAD channel: 2 = Z
N_EVAL    = 200                         # test traces used for evaluation (None = all)

# ---- study filter ----
# LARA only ever sees the first 1500 samples (15 s) of each 6000-sample trace, so the
# filter is chosen to keep the seismic signal inside that window. Measured on this
# STEAD build: P is inside the window for 100% of local events, but S leaves it at
# ~45-50 km. So the distance cap is the binding constraint, not the magnitude floor.
MIN_MAGNITUDE   = 2.5    # was 3.0 in the parent notebooks -> pool 14,983 -> 25,091
MAX_DISTANCE_KM = 60     # unchanged

# ---- study grid ----
CRS = [2, 3, 5, 10, 15, 20, 30, 50, 60, 100]

# BATCH   = 16
BATCH   = 64
WORKERS = 2
SEED    = 42

# ---- training recipes ----
MAIN_MODEL_NAME = "LARA"
LARA_VARIANT    = "pyramid_funnel32"    # LARA configuration inside class LARA
LARA_RECIPE     = "heavy"   # "heavy" = Final_HARA recipe (default) | "study" = LARA recipe
GRAD_CLIP       = 1.0

# "heavy" keeps the model in the exact slot the old HeavyAdvancedResidual occupied,
# so LARA stays directly comparable with the other models in this same run.
# "study" is the recipe LARA_27092026 was validated with (longer, higher LR, no WD).
RECIPES = {
    "heavy": {"max_epochs": 120, "lr": 2e-4, "weight_decay": 1e-5, "patience": 8, "sched_patience": 5},
    "study": {"max_epochs": 200, "lr": 5e-4, "weight_decay": 0.0,  "patience": 8, "sched_patience": 5},
}
GENERAL_MAX_EPOCHS = 100     # GeneralizedAutoencoder / AE_PureConcat

# ---- output ----
RESULTS_DIR = "results"
CKPT_DIR    = "ckpts"
SAVE_FIGS   = True

# ---- smoke mode overrides (applied before anything is created on disk) ----
if SMOKE:
    N_TRACES, CRS, N_EVAL = 100, [10], 10
    RESULTS_DIR, CKPT_DIR = "results_smoke", "ckpts_smoke"
    GENERAL_MAX_EPOCHS = 2
    for _recipe in RECIPES.values():
        _recipe["max_epochs"] = 2

# =============================
# Device selection and GPU setup
# =============================
def _backend_available(dev_type):
    if dev_type == "cpu":
        return True
    if dev_type == "cuda":
        return torch.cuda.is_available()
    if dev_type == "xpu":
        xpu = getattr(torch, "xpu", None)
        return xpu is not None and xpu.is_available()
    if dev_type == "mps":
        mps = getattr(torch.backends, "mps", None)
        return mps is not None and mps.is_available()
    return True


def resolve_device(override=None):
    """Pick the best available accelerator: CUDA -> XPU -> MPS -> CPU.

    Override with DEVICE_OVERRIDE to pin a specific device, e.g. "cuda:1".
    An override naming an unavailable backend fails fast here rather than
    part-way through training.
    """
    if override is None:
        override = DEVICE_OVERRIDE
    if override:
        dev = torch.device(override)
        if not _backend_available(dev.type):
            raise RuntimeError(
                f"DEVICE_OVERRIDE={override!r} but the {dev.type!r} backend is not "
                f"available in this environment (torch {torch.__version__}, "
                f"built with CUDA={torch.version.cuda}). "
                f"Set DEVICE_OVERRIDE = None to auto-detect, or 'cpu' to force CPU.")
        return dev
    if torch.cuda.is_available():
        return torch.device("cuda")
    xpu = getattr(torch, "xpu", None)
    if xpu is not None and xpu.is_available():
        return torch.device("xpu")
    mps = getattr(torch.backends, "mps", None)
    if mps is not None and mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


device = resolve_device()

if PIN_MEMORY is None:
    PIN_MEMORY = device.type == "cuda"

# cuDNN autotuning + TF32 only make sense on CUDA.
if device.type == "cuda":
    torch.backends.cudnn.benchmark = CUDNN_BENCHMARK
    torch.backends.cuda.matmul.allow_tf32 = ALLOW_TF32
    torch.backends.cudnn.allow_tf32 = ALLOW_TF32

AMP_DEVICE_TYPES = ("cuda", "xpu")
AMP_ENABLED = bool(USE_AMP) and device.type in AMP_DEVICE_TYPES


def print_device_report():
    """Print exactly which backend is in use, and why if it is not a GPU."""
    print("=" * 68)
    print("DEVICE REPORT")
    print("-" * 68)
    print(f"  torch version      : {torch.__version__}")
    print(f"  built with CUDA    : {torch.version.cuda}")
    print(f"  selected device    : {device}")
    print(f"  DEVICE_OVERRIDE    : {DEVICE_OVERRIDE}")

    if device.type == "cuda":
        idx = device.index if device.index is not None else torch.cuda.current_device()
        print(f"  GPU name           : {torch.cuda.get_device_name(idx)}")
        cap = torch.cuda.get_device_capability(idx)
        print(f"  compute capability : {cap[0]}.{cap[1]}")
        print(f"  visible GPUs       : {torch.cuda.device_count()}")
        print(f"  cudnn              : {torch.backends.cudnn.version()}")
        print(f"  cudnn.benchmark    : {torch.backends.cudnn.benchmark}")
        print(f"  TF32 allowed       : {ALLOW_TF32}")
    else:
        xpu = getattr(torch, "xpu", None)
        mps = getattr(torch.backends, "mps", None)
        print(f"  cuda available     : {torch.cuda.is_available()}")
        print(f"  xpu available      : {xpu.is_available() if xpu is not None else 'n/a'}")
        print(f"  mps available      : {mps.is_available() if mps is not None else 'n/a'}")
        print(f"  torch threads      : {torch.get_num_threads()} of {os.cpu_count()} cores")
        if torch.version.cuda is None:
            print("  NOTE: this torch build has no CUDA support at all (2.x+cpu wheel).")
            print("        On a CUDA machine install a CUDA wheel, e.g.")
            print("        pip install torch --index-url https://download.pytorch.org/whl/cu128")
        else:
            print("  NOTE: running on CPU - no accelerator was detected.")

    print(f"  AMP enabled        : {AMP_ENABLED}")
    print(f"  pin_memory         : {PIN_MEMORY}")
    print("=" * 68)


print_device_report()

results_dir = RESULTS_DIR
os.makedirs(results_dir, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

print("Results ->", os.path.abspath(results_dir))
print("Checkpoints ->", os.path.abspath(CKPT_DIR))
print("CRS:", CRS, "| batch:", BATCH, "| workers:", WORKERS)
print("Main model:", MAIN_MODEL_NAME, "| variant:", LARA_VARIANT, "| recipe:", LARA_RECIPE)
print(f"recipe: {RECIPES[LARA_RECIPE]}")
print("N_TRACES:", "ALL" if N_TRACES is None else N_TRACES, "| N_EVAL:", N_EVAL,
      "| SMOKE:", SMOKE)
print(f"Filter: earthquake_local, magnitude > {MIN_MAGNITUDE}, "
      f"distance_km <= {MAX_DISTANCE_KM}")


In [ ]:
# =============================
# Wavelet compression baselines
# =============================
WAVELET_TYPES = ['db4', 'sym8', 'coif3']

class WaveletCompressor:
    def __init__(self, wavelet_type, cr):
        self.wavelet_type = wavelet_type
        self.cr = cr
        
    def compress_decompress(self, signal):
        """Compress and decompress signal using wavelet thresholding"""
        try:
            # Perform wavelet decomposition
            coeffs = pywt.wavedec(signal, self.wavelet_type, level=6)
            
            # Calculate threshold based on compression ratio
            all_coeffs = np.concatenate([c.flatten() for c in coeffs])
            sorted_coeffs = np.sort(np.abs(all_coeffs))[::-1]
            
            # Keep only top coefficients to achieve target CR
            target_coeffs = len(signal) // self.cr
            if target_coeffs < 1:
                target_coeffs = 1
                
            if len(sorted_coeffs) > target_coeffs:
                threshold = sorted_coeffs[target_coeffs]
            else:
                threshold = 0
                
            # Apply threshold
            compressed_coeffs = []
            for coeff in coeffs:
                compressed_coeff = coeff * (np.abs(coeff) > threshold)
                compressed_coeffs.append(compressed_coeff)
            
            # Reconstruct signal
            reconstructed = pywt.waverec(compressed_coeffs, self.wavelet_type)
            
            # Ensure same length as original
            if len(reconstructed) > len(signal):
                reconstructed = reconstructed[:len(signal)]
            elif len(reconstructed) < len(signal):
                reconstructed = np.pad(reconstructed, (0, len(signal) - len(reconstructed)))
                
            return reconstructed
            
        except Exception as e:
            print(f"Wavelet compression error: {e}")
            return signal  # Return original if compression fails


class WaveletDB4:
    def __init__(self, cr):
        self.cr = cr
        self.wavelet_type = 'db4'
        
    def compress_decompress(self, signal):
        return WaveletCompressor('db4', self.cr).compress_decompress(signal)

class WaveletSYM8:
    def __init__(self, cr):
        self.cr = cr
        self.wavelet_type = 'sym8'
        
    def compress_decompress(self, signal):
        return WaveletCompressor('sym8', self.cr).compress_decompress(signal)

class WaveletCOIF3:
    def __init__(self, cr):
        self.cr = cr
        self.wavelet_type = 'coif3'
        
    def compress_decompress(self, signal):
        return WaveletCompressor('coif3', self.cr).compress_decompress(signal)

# =============================
# 🔹  TRAINING CONFIGURATION
# =============================


In [ ]:
# =============================
# Data: filtered STEAD pool -> 80/10/10 split (seed 42)
# =============================
# =============================
# Memory-efficient Dataset (per-trace min-max normalisation)
# =============================
class EfficientSeismicDataset(Dataset):
    def __init__(self, data):
        self.data = data
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        waveform = self.data[idx]
        z_min = np.min(waveform)
        z_max = np.max(waveform)
        
        if z_max - z_min > 0:
            waveform = (waveform - z_min) / (z_max - z_min)
        else:
            waveform = np.zeros_like(waveform)
        
        return torch.tensor(waveform, dtype=torch.float32).unsqueeze(0)


def load_filtered_stead(csv_path, h5_path, max_samples=None, component=COMPONENT):
    """Load the STEAD traces that pass the study filter, one component only.

    Filter: local earthquakes with source_magnitude > MIN_MAGNITUDE and
    source_distance_km <= MAX_DISTANCE_KM. Traces shorter than 1500 samples are
    skipped. Those two thresholds come from the CONFIG cell.
    """
    df = pd.read_csv(csv_path, low_memory=False,
                     usecols=["trace_name", "trace_category", "source_distance_km", "source_magnitude"])
    df = df[(df.trace_category == "earthquake_local") &
            (df.source_magnitude > MIN_MAGNITUDE) &
            (df.source_distance_km <= MAX_DISTANCE_KM)]
    names = df["trace_name"].to_list()
    if max_samples is not None:
        names = names[:max_samples]

    print(f"[data] filter: earthquake_local, magnitude > {MIN_MAGNITUDE}, "
          f"distance_km <= {MAX_DISTANCE_KM} -> pool = {len(df)}")
    print(f"[data] loading {len(names)} traces (component {component})")

    data = []
    with h5py.File(h5_path, "r") as f:
        for i, name in enumerate(names):
            try:
                w = f["data"][name][:]
                if w.shape[0] >= 1500:
                    data.append(w[:1500, component])
            except KeyError:
                continue
            if (i + 1) % 2000 == 0:
                print(f"  loaded {i + 1}/{len(names)}")

    print(f"[data] loaded waveforms: {len(data)}")
    return np.array(data)


waveforms = load_filtered_stead(CSV_PATH, H5_PATH, max_samples=N_TRACES, component=COMPONENT)
print(f"waveforms shape: {waveforms.shape}")

dataset = EfficientSeismicDataset(waveforms)
test_size  = int(0.1 * len(dataset))
val_size   = int(0.1 * len(dataset))
train_size = len(dataset) - test_size - val_size

# Stage 1: carve out the independent test set (10%)
train_val_dataset, test_dataset = train_test_split(dataset, test_size=test_size, random_state=SEED)
# Stage 2: split the rest 80/10 train/validation
train_dataset, val_dataset = train_test_split(train_val_dataset, train_size=train_size, random_state=SEED)

# pin_memory locks the host buffers so the CPU->GPU copy can be async (CUDA only)
train_loader = DataLoader(train_dataset, batch_size=BATCH, shuffle=True,  num_workers=WORKERS,
                          pin_memory=PIN_MEMORY)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH, shuffle=False, num_workers=WORKERS,
                          pin_memory=PIN_MEMORY)

# Evaluation subset of the held-out test split (never used for training or model selection)
n_eval      = len(test_dataset) if N_EVAL is None else min(N_EVAL, len(test_dataset))
test_data   = np.array([test_dataset[i] for i in range(n_eval)])
test_data_np = test_data[:, 0, :]

print(f"Full dataset size: {len(dataset)} waveforms")
print(f"Split 80/10/10 -> Train: {len(train_dataset)}, Val: {len(val_dataset)}, "
      f"Test: {len(test_dataset)}")
print(f"Evaluation subset (N_EVAL={N_EVAL}): {len(test_data)} of {len(test_dataset)} "
      f"test traces (test split used only for final evaluation)")


In [ ]:
# =============================
# 1- GeneralizedAutoencoder, 2- AE_PureConcat,
# shared AttentionBlock / EnhancedResidualBlock
# =============================
class GeneralizedAutoencoder(nn.Module):
    def __init__(self, cr):
        super().__init__()
        self.cr = cr
        latent_dim = 1500 // cr
        
        self.encoder = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=9, stride=2, padding=4), nn.ReLU(),
            nn.Conv1d(16, 32, kernel_size=9, stride=2, padding=4), nn.ReLU(),
            nn.Flatten(),
            nn.Linear(32 * 375, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 32 * 375),
            nn.Unflatten(1, (32, 375)),
            nn.ConvTranspose1d(32, 16, kernel_size=9, stride=2, padding=4, output_padding=1), nn.ReLU(),
            nn.ConvTranspose1d(16, 1, kernel_size=9, stride=2, padding=4, output_padding=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)

# -------------------------
# 🔹2- AE_PureConcat Model
# -------------------------
# Architecture configuration per compression ratio
model_2_architecture = {
    100: {"layers": 2, "feature_maps": [16, 16]},
     60: {"layers": 2, "feature_maps": [16, 16]},
     50: {"layers": 2, "feature_maps": [16, 16]},
     30: {"layers": 2, "feature_maps": [8,  8 ]},
     20: {"layers": 2, "feature_maps": [16, 16]},
     15: {"layers": 2, "feature_maps": [16, 16]},
     10: {"layers": 2, "feature_maps": [16, 16]},
      5: {"layers": 4, "feature_maps": [16, 16, 16, 16]},
      3: {"layers": 4, "feature_maps": [16, 16, 16, 16]},
      2: {"layers": 4, "feature_maps": [16, 16, 16, 16]},
}

class AE_PureConcat(nn.Module):
    def __init__(
        self,
        cr: int,
        input_len: int = 1500,
        kernel_size: int = 3,
        activation: nn.Module = nn.ELU,
        dropout: float = 0.1,
    ):
        super().__init__()
        cfg       = model_2_architecture[cr]
        layers    = cfg["layers"]
        fmaps     = cfg["feature_maps"]
        total_lat = input_len // cr

        # Distribute latent dimensions across branches
        base_dim  = total_lat // layers
        rem       = total_lat % layers
        latent_sizes = [base_dim + (1 if i < rem else 0) for i in range(layers)]

        # Encoder branches
        self.branches = nn.ModuleList()
        for i in range(layers):
            branch = nn.Sequential(
                nn.Conv1d(1, fmaps[i], kernel_size, padding=1),
                activation(inplace=True),
                nn.Dropout(dropout),
                nn.Flatten(),
                nn.Linear(input_len * fmaps[i], latent_sizes[i]),
                activation(inplace=True),
            )
            self.branches.append(branch)

        # Decoder: single linear layer to reconstruct full signal
        self.decoder = nn.Linear(total_lat, input_len)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x shape: (batch, 1, input_len)
        zs = [branch(x) for branch in self.branches]     # list of (batch, latent_i)
        z  = torch.cat(zs, dim=1)                        # (batch, total_lat)
        out = self.decoder(z)                            # (batch, input_len)
        return out.unsqueeze(1)                          # → (batch, 1, input_len)

# -------------------------
# 🔹3- HEAVY CAPACITY ARCHITECTURE (EXACT ABLATION STUDY PARAMETERS)
# -------------------------
class AttentionBlock(nn.Module):
    """Simplified channel attention mechanism - EXACT ABLATION VERSION"""
    def __init__(self, channels):
        super().__init__()
        self.attention = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Conv1d(channels, max(4, channels//8), kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv1d(max(4, channels//8), channels, kernel_size=1),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        attention_weights = self.attention(x)
        return x * attention_weights

class EnhancedResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, use_attention=False):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.elu = nn.ELU(inplace=True)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm1d(out_channels)
        
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm1d(out_channels)
            )
        
        self.attention = AttentionBlock(out_channels) if use_attention else nn.Identity()
            
    def forward(self, x):
        residual = self.shortcut(x)
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.elu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        out += residual
        out = self.elu(out)
        out = self.attention(out)
        return out


In [ ]:
# =============================
# Main model: LARA (Light Autoencoder for Reconstruction & Analysis)
# Replaces HeavyAdvancedResidualAutoencoder in the comparison
# =============================
class LARA(nn.Module):
    """
    LARA - Light Autoencoder for Reconstruction & Analysis.

    Single production configuration: variant='pyramid_funnel32' (see VARIANT in the CONFIG
    cell). Deep residual + channel-attention encoder, filled projection funnel, learned
    channel-pyramid decoder, conv refinement head.

    variant='pyramid_funnel32' (LARA): FILLS THE FUNNEL - T = L (no min(latent, L) clamp),
                               so the projection width no longer collapses as CR grows, and
                               BOTTLE_CHANNELS=32 -> 6016 projection columns (cr<=30) /
                               3008 (cr>30). cr<10 is identical to 'pyramid'.
    Compression ratio is preserved exactly: latent == input_len // cr (asserted).
    MSE loss only -> fair against the verbatim baseline recipe.

    Other internal tokens kept for provenance / ablations:
    variant='pyramid'   : single latent path; channel pyramid (16->32->64) + refinement head.
    variant='pyramid_v1': exact original 'pyramid' behaviour (archival alias).
    variant='pyramid_b32'    : 32-channel bottleneck spine for cr>=10 (16 for cr<10).
    variant='pyramid_b32_hd' : b32 + high-CR decoder polish for cr>=30 (ceiling 96, k9).
    variant='multiscale': latent split coarse (ceil half) + detail (rest), finest feature
                          carried in its own code, both decoded and fused.
    variant='pyramid_funnel64': same T = L fix with B=64 (cr<=30) / 128 (cr>30) -> 12032
                               columns at EVERY CR (baseline-parity width).
    """

    BOTTLE_CHANNELS = 16

    def __init__(self, cr, input_len=1500, variant="pyramid"):
        super().__init__()
        self.cr = cr
        self.input_len = input_len
        self.variant = variant
        self.latent_dim = input_len // cr

        # ---- variant knobs ----
        b32 = variant in ("pyramid_b32", "pyramid_b32_hd") and cr >= 10
        hd = variant == "pyramid_b32_hd" and cr >= 30
        funnel = variant in ("pyramid_funnel32", "pyramid_funnel64") and cr >= 10
        if variant == "pyramid_funnel32" and funnel:
            self.BOTTLE_CHANNELS = 32
        elif variant == "pyramid_funnel64" and funnel:
            self.BOTTLE_CHANNELS = 64 if cr <= 30 else 128
        else:
            self.BOTTLE_CHANNELS = 32 if b32 else 16
        self.pyramid_ceiling = max(64, self.BOTTLE_CHANNELS) if funnel else (96 if hd else 64)
        self.final_kernel = 9 if hd else 5

        self.L = 188 if cr <= 30 else 94
        self.T = self.L if funnel else min(self.latent_dim, self.L)

        if variant == "multiscale":
            self.latent_coarse = (self.latent_dim + 1) // 2
            self.latent_detail = self.latent_dim - self.latent_coarse
            self.T2 = min(32, max(2, self.latent_dim // 8))
        else:
            self.latent_coarse = self.latent_dim
            self.latent_detail = 0
            self.T2 = None
        assert self.latent_coarse + self.latent_detail == self.latent_dim, "CR must stay exact"

        # ---- encoder stages ----
        self.stage0 = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=15, stride=2, padding=7),
            nn.BatchNorm1d(32),
            nn.ELU(inplace=True),
            AttentionBlock(32),
        )
        self.stage1 = EnhancedResidualBlock(32, 64, stride=2, use_attention=True)
        self.stage2 = EnhancedResidualBlock(64, 128, stride=2, use_attention=True)
        self.stage3 = EnhancedResidualBlock(128, 256, stride=2, use_attention=True) if cr > 30 else None
        final_channels = 256 if self.stage3 is not None else 128

        # ---- coarse bottleneck (original Light path) ----
        self.bot_conv = nn.Sequential(
            nn.Conv1d(final_channels, self.BOTTLE_CHANNELS, kernel_size=1),
            nn.ELU(inplace=True),
        )
        self.bot_pool = nn.Sequential(
            nn.AdaptiveAvgPool1d(self.T),
            nn.Flatten(),
        )
        self.coarse_head = nn.Linear(self.BOTTLE_CHANNELS * self.T, self.latent_coarse)

        # ---- detail bottleneck (multiscale only): finest 32ch @ 750 feature ----
        self.detail_block = nn.Sequential()
        self.detail_head = nn.Identity()
        if variant == "multiscale":
            self.detail_block = nn.Sequential(
                nn.Conv1d(32, 32, kernel_size=3, padding=1),
                nn.BatchNorm1d(32),
                nn.ELU(inplace=True),
                nn.AdaptiveAvgPool1d(self.T2),
                nn.Flatten(),
            )
            self.detail_head = nn.Linear(32 * self.T2, self.latent_detail)

        # ---- decoders (channel pyramid + learned upsampling) ----
        self.coarse_decoder = nn.Sequential(
            nn.Linear(self.latent_coarse, self.BOTTLE_CHANNELS * self.T),
            nn.Unflatten(1, (self.BOTTLE_CHANNELS, self.T)),
            nn.ELU(inplace=True),
            self._pyramid(self.BOTTLE_CHANNELS, self.T),
        )
        self.detail_decoder = nn.Sequential()
        if variant == "multiscale":
            self.detail_decoder = nn.Sequential(
                nn.Linear(self.latent_detail, self.BOTTLE_CHANNELS * self.T2),
                nn.Unflatten(1, (self.BOTTLE_CHANNELS, self.T2)),
                nn.ELU(inplace=True),
                self._pyramid(self.BOTTLE_CHANNELS, self.T2),
            )

        # ---- refinement head ----
        in_ch = 128 if variant == "multiscale" else self.pyramid_ceiling
        self.head = nn.Sequential(
            nn.Conv1d(in_ch, 32, kernel_size=5, padding=2),
            nn.ELU(inplace=True),
            nn.Conv1d(32, 1, kernel_size=15, padding=7),
            nn.Sigmoid(),
        )

        self.test_dimensions()

    # ------------------------------------------------------------------ #
    # encoding / decoding
    # ------------------------------------------------------------------ #
    def encoder(self, x):
        """Concatenated latent (coarse + detail) - the actual transmitted code."""
        zc = self.encode_coarse(x)
        if self.variant == "multiscale":
            zd = self.encode_detail(x)
            return torch.cat([zc, zd], dim=-1)
        return zc

    def encode_coarse(self, x):
        h = self.stage0(x)
        h = self.stage1(h)
        h = self.stage2(h)
        if self.stage3 is not None:
            h = self.stage3(h)
        f = self.bot_conv(h)
        f = self.bot_pool(f)
        return self.coarse_head(f)

    def encode_detail(self, x):
        f = self.stage0(x)
        d = self.detail_block(f)
        return self.detail_head(d)

    def forward(self, x):
        zc = self.encode_coarse(x)
        cf = self.coarse_decoder(zc)
        if self.variant == "multiscale":
            zd = self.encode_detail(x)
            df = self.detail_decoder(zd)
            fused = torch.cat([cf, df], dim=1)
            return self.head(fused)
        return self.head(cf)

    # ------------------------------------------------------------------ #
    # helpers
    # ------------------------------------------------------------------ #
    def _pyramid(self, start_channels, start_len):
        """Channel-pyramid upsampler from start_len to input_len using learned
        (nearest + conv) upsampling. Channel ramp up to the variant ceiling."""
        nlev = 6
        ratio = self.input_len / start_len
        levels = []
        cch = start_channels
        prev_len = start_len
        for k in range(nlev):
            tgt = int(round(start_len * (ratio ** ((k + 1) / nlev))))
            tgt = max(tgt, prev_len + 1)
            if k == nlev - 1:
                tgt = self.input_len
            ksize = self.final_kernel if k == nlev - 1 else 5
            nc = min(self.pyramid_ceiling, int(cch * 2))
            levels.append(nn.Sequential(
                nn.Upsample(size=tgt, mode='nearest'),
                nn.Conv1d(cch, nc, kernel_size=ksize, padding=ksize // 2, bias=False),
                nn.BatchNorm1d(nc),
                nn.ELU(inplace=True),
            ))
            cch = nc
            prev_len = tgt
        return nn.Sequential(*levels)

    def test_dimensions(self):
        try:
            with torch.no_grad():
                test_input = torch.randn(1, 1, self.input_len)
                zc = self.encode_coarse(test_input)
                assert zc.numel() == self.latent_coarse, f"coarse latent {zc.numel()} != {self.latent_coarse}"
                if self.variant == "multiscale":
                    zd = self.encode_detail(test_input)
                    assert zd.numel() == self.latent_detail, f"detail latent {zd.numel()} != {self.latent_detail}"
                total = self.encoder(test_input).numel()
                assert total == self.latent_dim, f"total latent {total} != {self.latent_dim}"
                decoded = self(test_input)
                assert decoded.shape[2] == self.input_len, f"output {decoded.shape[2]} != {self.input_len}"
                actual_cr = self.input_len / total
                print(f"[OK] Light-HARA-V2 ({self.variant}) CR={self.cr}: Input {self.input_len} -> Latent {total} "
                      f"(coarse {self.latent_coarse} + detail {self.latent_detail}) -> Output {decoded.shape[2]} "
                      f"(True CR = {actual_cr:.2f})")
        except Exception as e:
            print(f"[FAIL] Light-HARA-V2 ({self.variant}) CR={self.cr}: {e}")
            self.create_fallback_architecture()

    def create_fallback_architecture(self):
        print("Creating fallback architecture...")
        self.stage0 = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=15, stride=2, padding=7), nn.ELU(),
            nn.Conv1d(32, 64, kernel_size=9, stride=2, padding=4), nn.ELU(),
        )
        self.stage1 = nn.Identity()
        self.stage2 = nn.Identity()
        self.stage3 = None
        self.bot_conv = nn.Conv1d(64, self.BOTTLE_CHANNELS, kernel_size=1)
        self.bot_pool = nn.Sequential(nn.AdaptiveAvgPool1d(self.T), nn.Flatten())
        self.coarse_head = nn.Linear(self.BOTTLE_CHANNELS * self.T, self.latent_dim)
        self.latent_coarse = self.latent_dim
        self.latent_detail = 0
        self.variant = "pyramid"
        self.detail_block = nn.Sequential()
        self.detail_head = nn.Identity()
        self.coarse_decoder = nn.Sequential(
            nn.Linear(self.latent_dim, self.BOTTLE_CHANNELS * self.T),
            nn.Unflatten(1, (self.BOTTLE_CHANNELS, self.T)), nn.ELU(),
            nn.Upsample(size=self.input_len, mode='linear', align_corners=False),
            nn.Conv1d(self.BOTTLE_CHANNELS, 1, kernel_size=15, padding=7), nn.Sigmoid(),
        )
        self.detail_decoder = nn.Sequential()
        self.head = nn.Identity()
        print("[OK] Fallback architecture created")


In [ ]:
# =============================
# Evaluation metrics, model size, checkpoint paths
# =============================
def calculate_psnr(original, reconstructed):
    """Calculate PSNR between original and reconstructed signals"""
    mse = np.mean((original - reconstructed) ** 2)
    if mse == 0:
        return float('inf')
    max_pixel = 1.0
    return 20 * math.log10(max_pixel / math.sqrt(mse))


def calculate_ssim_1d(original, reconstructed, data_range=1.0):
    """Calculate SSIM for 1D signals"""
    if len(original.shape) > 1:
        original = original.flatten()
    if len(reconstructed.shape) > 1:
        reconstructed = reconstructed.flatten()

    return ssim(original, reconstructed, data_range=data_range)


def calculate_model_size(model):
    """Calculate model size in MB"""
    param_size = 0
    for param in model.parameters():
        param_size += param.nelement() * param.element_size()
    buffer_size = 0
    for buffer in model.buffers():
        buffer_size += buffer.nelement() * buffer.element_size()

    return (param_size + buffer_size) / 1024 ** 2


def best_file(model_name, cr):
    """Checkpoint path for a (model, cr) pair - single accessor for save/load/cleanup."""
    return os.path.join(CKPT_DIR, f"best_{model_name}_cr{cr}.pth")


In [ ]:
# =============================
# TRAINING CONFIGURATION
# =============================
def _make_scaler():
    """GradScaler for mixed precision, inactive unless AMP_ENABLED."""
    return torch.amp.GradScaler(device.type, enabled=AMP_ENABLED)


def train_model_main(model, train_loader, val_loader, model_name, cr,
                     recipe=None, max_epochs=None):
    """Trainer for the main model (LARA).

    `recipe` selects a hyper-parameter set from RECIPES:
      "heavy" - the recipe Final_HARA_20092026 used for HeavyAdvancedResidual
                (Adam 2e-4, wd 1e-5, 120 epochs, early-stop patience 8)
      "study" - the recipe LARA_27092026 was validated with
                (Adam 5e-4, no weight decay, 200 epochs, sched patience 5)
    Uniform epoch budget; early stopping decides the effective duration.
    """
    recipe = recipe or LARA_RECIPE
    cfg = dict(RECIPES[recipe])
    if max_epochs is not None:
        cfg["max_epochs"] = max_epochs

    model = model.to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(),
                                 lr=cfg["lr"],
                                 weight_decay=cfg["weight_decay"])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=cfg["sched_patience"], min_lr=1e-6)
    scaler = _make_scaler()

    best_val_loss = float('inf')
    patience_counter = 0
    train_losses, val_losses = [], []

    ckpt = best_file(model_name, cr)
    print(f"Training {model_name} CR={cr} on {device} - recipe '{recipe}' "
          f"{cfg} (uniform {cfg['max_epochs']}-epoch budget, early stopping decides, "
          f"amp={AMP_ENABLED})")
    t_start = time.time()

    for epoch in range(cfg["max_epochs"]):
        model.train()
        train_loss = 0.0

        for batch_idx, batch in enumerate(train_loader):
            inputs = batch.to(device, non_blocking=PIN_MEMORY)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, enabled=AMP_ENABLED):
                outputs = model(inputs)
                loss = criterion(outputs, inputs)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
            train_loss += loss.item()

            if batch_idx % 500 == 0:
                print(f"Epoch {epoch + 1}, Batch {batch_idx}, Loss: {loss.item():.6f}")

        # validation runs in full precision so early stopping compares like with like
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch in val_loader:
                inputs = batch.to(device, non_blocking=PIN_MEMORY)
                outputs = model(inputs)
                val_loss += criterion(outputs, inputs).item()

        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)

        train_losses.append(avg_train_loss)
        val_losses.append(avg_val_loss)
        scheduler.step(avg_val_loss)

        print(f"Epoch {epoch + 1}/{cfg['max_epochs']} - Train Loss: {avg_train_loss:.6f}, "
              f"Val Loss: {avg_val_loss:.6f}, LR: {optimizer.param_groups[0]['lr']:.6f}")

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            patience_counter = 0
            torch.save(model.state_dict(), ckpt)
            print(f"\N{HEAVY CHECK MARK} New best model saved (val_loss: {best_val_loss:.6f})")
        else:
            patience_counter += 1
            if patience_counter >= cfg["patience"]:
                print(f"Early stopping at epoch {epoch + 1}")
                break

        if epoch % 2 == 0 and device.type in AMP_DEVICE_TYPES:
            torch.cuda.empty_cache() if device.type == "cuda" else torch.xpu.empty_cache()

    # Reload the best weights from disk. The checkpoint is written at the epoch with the
    # lowest validation loss, so this is the correct model to evaluate - and it avoids
    # holding a second full copy of the weights in memory.
    if os.path.exists(ckpt):
        model.load_state_dict(torch.load(ckpt, map_location=device))
        print(f"\N{HEAVY CHECK MARK} Loaded best model from checkpoint (val_loss: {best_val_loss:.6f})")
    else:
        print("No checkpoint was written - keeping the final-epoch weights")

    print(f"Training {model_name} CR={cr} finished in {(time.time() - t_start) / 60.0:.1f} min")
    return model, train_losses, val_losses


def train_model_general(model, train_loader, val_loader, model_name, cr, max_epochs=None):
    """General training function for the smaller baseline models"""
    if max_epochs is None:
        max_epochs = GENERAL_MAX_EPOCHS

    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)
    scaler = _make_scaler()

    best_val_loss = float('inf')
    patience = 8
    patience_counter = 0

    train_losses, val_losses = [], []
    ckpt = best_file(model_name, cr)

    print(f"Training {model_name} CR={cr} on {device} (max_epochs={max_epochs}, amp={AMP_ENABLED})")
    t_start = time.time()

    for epoch in range(max_epochs):
        model.train()
        train_loss = 0.0

        for batch in train_loader:
            inputs = batch.to(device, non_blocking=PIN_MEMORY)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, enabled=AMP_ENABLED):
                outputs = model(inputs)
                loss = nn.MSELoss()(inputs, outputs)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            train_loss += loss.item()

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch in val_loader:
                inputs = batch.to(device, non_blocking=PIN_MEMORY)
                outputs = model(inputs)
                val_loss += nn.MSELoss()(inputs, outputs).item()

        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)

        train_losses.append(avg_train_loss)
        val_losses.append(avg_val_loss)
        scheduler.step(avg_val_loss)

        print(f"Epoch {epoch + 1}/{max_epochs} - Train Loss: {avg_train_loss:.6f}, "
              f"Val Loss: {avg_val_loss:.6f}, LR: {optimizer.param_groups[0]['lr']:.6f}")

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            patience_counter = 0
            torch.save(model.state_dict(), ckpt)
            print(f"\N{HEAVY CHECK MARK} New best model saved (val_loss: {best_val_loss:.6f})")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch + 1}")
                break

    if os.path.exists(ckpt):
        model.load_state_dict(torch.load(ckpt, map_location=device))
        print(f"\N{HEAVY CHECK MARK} Loaded best model from checkpoint (val_loss: {best_val_loss:.6f})")
    else:
        print("No checkpoint was written - keeping the final-epoch weights")

    print(f"Training {model_name} CR={cr} finished in {(time.time() - t_start) / 60.0:.1f} min")
    return model, train_losses, val_losses


In [ ]:
# =============================
# Model evaluation (neural + wavelet baselines)
# =============================
def evaluate_model(model, test_data, cr, model_name):
    """Per-sample evaluation, so that mean / std / median / quartiles are meaningful.

    Wavelet baselines receive 2-D (N, 1500) data and are run through
    `compress_decompress` one trace at a time; neural models receive 3-D
    (N, 1, 1500) data. Wavelet models have no learnable parameters, so their
    Parameter_Count and Model_Size_MB are reported as 0.
    """
    is_wavelet = "Wavelet" in model_name

    if is_wavelet:
        x = np.asarray(test_data)
        recon = []
        for signal in x:
            signal_1d = signal[0] if len(signal.shape) > 1 else signal
            recon.append(model.compress_decompress(signal_1d))
        recon = np.asarray(recon)
        x_processed = x[:, 0, :] if len(x.shape) > 2 else x
    else:
        model.eval()
        with torch.no_grad():
            inputs = torch.tensor(test_data, dtype=torch.float32).to(device,
                                                                    non_blocking=PIN_MEMORY)
            recon = model(inputs).cpu().numpy()
            x_processed = inputs.cpu().numpy()[:, 0, :]

    if recon.ndim == 3:
        recon = recon[:, 0, :]

    n_samples = x_processed.shape[0]
    mse_list, snr_list, corr_list, mae_list, psnr_list, ssim_list, grad_l1_list = [], [], [], [], [], [], []

    for i in range(n_samples):
        orig = x_processed[i]
        rec = recon[i]
        m = np.mean((orig - rec) ** 2)
        signal_power = np.mean(orig ** 2)
        snr_i = 10 * math.log10(signal_power / (m + 1e-12)) if m > 0 else float('inf')
        if np.std(orig) > 0 and np.std(rec) > 0:
            corr_i = np.corrcoef(orig, rec)[0, 1]
        else:
            corr_i = 0.0
        mse_list.append(m)
        snr_list.append(snr_i)
        corr_list.append(corr_i)
        mae_list.append(np.mean(np.abs(orig - rec)))
        psnr_list.append(calculate_psnr(orig, rec))
        ssim_list.append(calculate_ssim_1d(orig, rec))
        grad_l1_list.append(float(np.mean(np.abs(np.diff(orig) - np.diff(rec)))))

    snr_values = np.asarray(snr_list, dtype=float)
    snr_finite = snr_values[np.isfinite(snr_values)]
    snr_mean    = float(np.mean(snr_finite)) if snr_finite.size > 0 else float('inf')
    snr_std     = float(np.std(snr_finite)) if snr_finite.size > 0 else 0.0
    snr_median  = float(np.median(snr_finite)) if snr_finite.size > 0 else float('inf')
    snr_p25     = float(np.percentile(snr_finite, 25)) if snr_finite.size > 0 else float('inf')
    snr_p75     = float(np.percentile(snr_finite, 75)) if snr_finite.size > 0 else float('inf')

    ssim_values = np.asarray(ssim_list, dtype=float)

    # ---- achieved compression ratio ----
    if is_wavelet:
        actual_cr = cr
    else:
        test_input = torch.randn(1, 1, 1500).to(device)
        compressed = model.encoder(test_input) if hasattr(model, 'encoder') else None
        # AE_PureConcat exposes no .encoder -> fall back to the nominal CR
        actual_cr = 1500 / (compressed.numel() // compressed.shape[0]) if compressed is not None else cr

    cr_error = abs(actual_cr - cr) / cr * 100

    # ---- complexity (neural models only) ----
    if is_wavelet:
        model_size = 0.0
        total_params = 0
    else:
        model_size = calculate_model_size(model)
        total_params = sum(p.numel() for p in model.parameters())

    return {
        "MSE": float(np.mean(mse_list)),
        "MSE_std": float(np.std(mse_list)),
        "SNR": snr_mean,
        "SNR_std": snr_std,
        "SNR_median": snr_median,
        "SNR_P25": snr_p25,
        "SNR_P75": snr_p75,
        "Correlation": float(np.mean(corr_list)),
        "Correlation_std": float(np.std(corr_list)),
        "MAE": float(np.mean(mae_list)),
        "MAE_std": float(np.std(mae_list)),
        "PSNR": float(np.mean(psnr_list)),
        "PSNR_std": float(np.std(psnr_list)),
        "SSIM": float(np.mean(ssim_values)),
        "SSIM_std": float(np.std(ssim_values)),
        "SSIM_median": float(np.median(ssim_values)),
        "SSIM_P25": float(np.percentile(ssim_values, 25)),
        "SSIM_P75": float(np.percentile(ssim_values, 75)),
        "GradL1": float(np.mean(grad_l1_list)),
        "GradL1_std": float(np.std(grad_l1_list)),
        "SNR_samples": snr_finite.tolist(),
        "CR_Error": cr_error,
        "Model_Size_MB": model_size,
        "Parameter_Count": total_params,
        "Model": model_name,
        "CR": cr,
        "Actual_CR": actual_cr,
        "Reconstructions": (x_processed, recon),
    }


# =============================
# Statistical significance testing
# =============================
def statistical_significance_test(results_df, metric='SNR'):
    """Independent t-test between every pair of models.

    NOTE: this compares the per-CR summary values, not per-trace values, so the
    p-values are weak evidence. Pairs where either model has fewer than 2 CRs are
    reported as nan rather than silently producing a divide-by-zero.
    """
    models = results_df['Model'].unique()
    p_values = {}

    for i, model1 in enumerate(models):
        for model2 in models[i + 1:]:
            data1 = results_df[results_df['Model'] == model1][metric]
            data2 = results_df[results_df['Model'] == model2][metric]

            if len(data1) < 2 or len(data2) < 2:
                p_values[f"{model1}_vs_{model2}"] = float('nan')
                print(f"  skip {model1}_vs_{model2}: needs >=2 CRs per model (have {len(data1)}, {len(data2)})")
                continue

            _, p_value = stats.ttest_ind(data1, data2)
            p_values[f"{model1}_vs_{model2}"] = p_value

    return p_values


In [ ]:
# =============================
# Visualisation, learning curves & parameter analysis
# =============================
def plot_waveforms_comprehensive(original, reconstructed, cr, model_name, save_figures=True):
    """
    Enhanced visualization with time and frequency domains for ALL neural network models
    """
    num_samples = min(6, len(original))
    
    # Create figure with subplots
    fig, axes = plt.subplots(num_samples, 2, figsize=(15, 3*num_samples))
    
    # Handle single sample case
    if num_samples == 1:
        axes = np.array([axes])
    
    for i in range(num_samples):
        # Ensure we have 1D arrays for processing
        orig_signal = original[i].flatten() if len(original[i].shape) > 1 else original[i]
        recon_signal = reconstructed[i].flatten() if len(reconstructed[i].shape) > 1 else reconstructed[i]
        
        # Ensure both signals have the same length (1500)
        min_len = min(len(orig_signal), len(recon_signal))
        orig_signal = orig_signal[:min_len]
        recon_signal = recon_signal[:min_len]
        
        # Time domain plot
        axes[i, 0].plot(orig_signal, 'b-', label='Original', linewidth=1.5, alpha=0.8)
        axes[i, 0].plot(recon_signal, 'r-', label='Reconstructed', linewidth=1.5, alpha=0.8)
        axes[i, 0].set_title(f'{model_name} - CR={cr} - Sample {i+1}', fontsize=12, fontweight='bold')
        axes[i, 0].set_xlabel('Time Samples')
        axes[i, 0].set_ylabel('Amplitude (Normalized)')
        axes[i, 0].legend()
        axes[i, 0].grid(True, alpha=0.3)
        
        # Frequency domain plot - FIXED DIMENSIONALITY ISSUE
        try:
            orig_fft = np.abs(np.fft.fft(orig_signal))
            recon_fft = np.abs(np.fft.fft(recon_signal))
            freq = np.fft.fftfreq(len(orig_signal))
            
            # Only plot positive frequencies
            positive_freq_mask = freq >= 0
            freq_positive = freq[positive_freq_mask]
            orig_fft_positive = orig_fft[positive_freq_mask]
            recon_fft_positive = recon_fft[positive_freq_mask]
            
            axes[i, 1].semilogy(freq_positive, orig_fft_positive, 'b-', label='Original', linewidth=1.5, alpha=0.8)
            axes[i, 1].semilogy(freq_positive, recon_fft_positive, 'r-', label='Reconstructed', linewidth=1.5, alpha=0.8)
            axes[i, 1].set_title('Frequency Spectrum', fontsize=12, fontweight='bold')
            axes[i, 1].set_xlabel('Frequency (Normalized)')
            axes[i, 1].set_ylabel('Magnitude (log scale)')
            axes[i, 1].legend()
            axes[i, 1].grid(True, alpha=0.3)
        except Exception as e:
            print(f"Warning: Frequency plot failed for sample {i+1}: {e}")
            axes[i, 1].text(0.5, 0.5, f'Frequency plot failed\n{str(e)}', 
                           ha='center', va='center', transform=axes[i, 1].transAxes)
            axes[i, 1].set_title('Frequency Spectrum - Failed', fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    
    if save_figures:
        # Create model-specific directory
        model_dir = os.path.join(results_dir, model_name)
        os.makedirs(model_dir, exist_ok=True)
        
        # Save figures in multiple formats
        filename_base = f"{model_name}_CR{cr}_comprehensive"
        png_path = os.path.join(model_dir, f"{filename_base}.png")
        eps_path = os.path.join(model_dir, f"{filename_base}.eps")
        
        plt.savefig(png_path, dpi=300, bbox_inches='tight', facecolor='white')
        plt.savefig(eps_path, format='eps', bbox_inches='tight', facecolor='white')
        print(f"✓ Saved comprehensive waveforms to: {png_path} and {eps_path}")
    
    plt.show()
    plt.close()

def plot_frequency_spectrum_all_models(original, reconstructed, cr, model_name, save_figures=True):
    """
    Save frequency spectrum plots for ALL models (including autoencoders) for first six samples
    """
    num_samples = min(6, len(original))
    
    for i in range(num_samples):
        try:
            # Ensure we have 1D arrays for processing
            orig_signal = original[i].flatten() if len(original[i].shape) > 1 else original[i]
            recon_signal = reconstructed[i].flatten() if len(reconstructed[i].shape) > 1 else reconstructed[i]
            
            # Ensure both signals have the same length
            min_len = min(len(orig_signal), len(recon_signal))
            orig_signal = orig_signal[:min_len]
            recon_signal = recon_signal[:min_len]
            
            # Create figure for frequency spectrum only
            plt.figure(figsize=(10, 6))
            
            # Calculate frequency spectra
            orig_fft = np.abs(np.fft.fft(orig_signal))
            recon_fft = np.abs(np.fft.fft(recon_signal))
            freq = np.fft.fftfreq(len(orig_signal))
            
            # Only plot positive frequencies
            positive_freq_mask = freq >= 0
            freq_positive = freq[positive_freq_mask]
            orig_fft_positive = orig_fft[positive_freq_mask]
            recon_fft_positive = recon_fft[positive_freq_mask]
            
            # Plot frequency spectrum
            plt.semilogy(freq_positive, orig_fft_positive, 'b-', label='Original', linewidth=2, alpha=0.8)
            plt.semilogy(freq_positive, recon_fft_positive, 'r-', label='Reconstructed', linewidth=2, alpha=0.8)
            plt.title(f'{model_name} - CR={cr} - Sample {i+1} - Frequency Spectrum', fontsize=14, fontweight='bold')
            plt.xlabel('Frequency (Normalized)', fontsize=12)
            plt.ylabel('Magnitude (log scale)', fontsize=12)
            plt.legend(fontsize=11)
            plt.grid(True, alpha=0.3)
            
            if save_figures:
                # Create frequency spectrum directory
                freq_dir = os.path.join(results_dir, model_name, "frequency_spectrum")
                os.makedirs(freq_dir, exist_ok=True)
                
                # Save in both PNG and EPS formats
                filename_base = f"{model_name}_CR{cr}_sample{i+1}_freq_spectrum"
                png_path = os.path.join(freq_dir, f"{filename_base}.png")
                eps_path = os.path.join(freq_dir, f"{filename_base}.eps")
                
                plt.savefig(png_path, dpi=300, bbox_inches='tight', facecolor='white')
                plt.savefig(eps_path, format='eps', bbox_inches='tight', facecolor='white')
                print(f"✓ Saved frequency spectrum for sample {i+1} to: {png_path}")
            
            plt.close()
            
        except Exception as e:
            print(f"Error creating frequency spectrum for {model_name} CR={cr} sample {i+1}: {e}")
            plt.close()

def plot_training_history(train_losses, val_losses, model_name, cr, save_figures=True):
    """Plot training and validation loss history"""
    if not train_losses or len(train_losses) == 0:
        print(f"No training history available for {model_name} CR={cr}")
        return
        
    plt.figure(figsize=(12, 8))
    epochs = range(1, len(train_losses) + 1)
    
    plt.plot(epochs, train_losses, 'b-', label='Training Loss', linewidth=2, alpha=0.8)
    plt.plot(epochs, val_losses, 'r-', label='Validation Loss', linewidth=2, alpha=0.8)
    
    plt.title(f'{model_name} - CR={cr} - Training History', fontsize=16, fontweight='bold')
    plt.xlabel('Epoch', fontsize=14)
    plt.ylabel('MSE Loss', fontsize=14)
    plt.yscale('log')
    plt.legend(fontsize=12)
    plt.grid(True, alpha=0.3)
    
    # Add final loss values to the plot
    final_train_loss = train_losses[-1]
    final_val_loss = val_losses[-1] if val_losses else train_losses[-1]
    plt.annotate(f'Final Train: {final_train_loss:.6f}\nFinal Val: {final_val_loss:.6f}', 
                xy=(0.02, 0.98), xycoords='axes fraction',
                bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", alpha=0.8),
                fontsize=10, ha='left', va='top')
    
    if save_figures:
        model_dir = os.path.join(results_dir, model_name)
        os.makedirs(model_dir, exist_ok=True)
        
        filename_base = f"{model_name}_CR{cr}_training_history"
        png_path = os.path.join(model_dir, f"{filename_base}.png")
        eps_path = os.path.join(model_dir, f"{filename_base}.eps")
        
        plt.savefig(png_path, dpi=300, bbox_inches='tight')
        plt.savefig(eps_path, format='eps', bbox_inches='tight')
        print(f"✓ Saved training history to: {png_path}")
    
    plt.show()
    plt.close()


def plot_parameter_comparison(results_df, save_figures=True):
    """Plot parameter count comparison across all models and compression ratios"""
    neural_models = list(results_df['Model'].unique())
    cr_values = sorted(results_df['CR'].unique())
    
    plt.figure(figsize=(14, 8))
    
    # Create color map for models
    colors = plt.cm.Set3(np.linspace(0, 1, len(neural_models)))
    
    for i, model in enumerate(neural_models):
        param_counts = []
        for cr in cr_values:
            model_cr_data = results_df[(results_df['Model'] == model) & (results_df['CR'] == cr)]
            if not model_cr_data.empty:
                param_counts.append(model_cr_data['Parameter_Count'].values[0] / 1e6)
            else:
                param_counts.append(np.nan)
        
        plt.semilogx(cr_values, param_counts, 'o-', color=colors[i], label=model, 
                    markersize=8, linewidth=2, markerfacecolor=colors[i], markeredgecolor='black')
    
    plt.title('Parameter Count Comparison Across Models', fontsize=16, fontweight='bold')
    plt.xlabel('Compression Ratio', fontsize=14)
    plt.ylabel('Parameter Count (Millions)', fontsize=14)
    plt.grid(True, alpha=0.3)
    plt.legend(fontsize=12, loc='upper right')
    
    # Add text annotation with model statistics
    stats_text = "Model Parameter Statistics:\n"
    for model in neural_models:
        model_data = results_df[results_df['Model'] == model]
        avg_params = model_data['Parameter_Count'].mean() / 1e6
        max_params = model_data['Parameter_Count'].max() / 1e6
        min_params = model_data['Parameter_Count'].min() / 1e6
        stats_text += f"{model}: {avg_params:.2f}M avg\n"
    
    plt.annotate(stats_text, xy=(0.02, 0.02), xycoords='axes fraction',
                bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", alpha=0.8),
                fontsize=10, ha='left', va='bottom')
    
    if save_figures:
        analysis_dir = os.path.join(results_dir, "model_analysis")
        os.makedirs(analysis_dir, exist_ok=True)
        
        png_path = os.path.join(analysis_dir, "parameter_count_comparison.png")
        eps_path = os.path.join(analysis_dir, "parameter_count_comparison.eps")
        
        plt.savefig(png_path, dpi=300, bbox_inches='tight')
        plt.savefig(eps_path, format='eps', bbox_inches='tight')
        print(f"✓ Saved parameter count comparison to: {png_path}")
    
    plt.show()
    plt.close()

def plot_parameter_vs_performance(results_df, save_figures=True):
    """Plot parameter count vs performance metrics"""
    neural_models = list(results_df['Model'].unique())
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    axes = axes.flatten()
    
    metrics = ['SNR', 'PSNR', 'SSIM', 'MSE']
    
    for i, metric in enumerate(metrics):
        for model in neural_models:
            model_data = results_df[results_df['Model'] == model]
            if not model_data.empty:
                params = model_data['Parameter_Count'] / 1e6
                performance = model_data[metric]
                
                markers = ['o', 's', 'D', '^', 'v', '<', '>']
                marker = markers[neural_models.index(model) % len(markers)]
                
                axes[i].scatter(params, performance, label=model, marker=marker, s=80, alpha=0.7)
        
        axes[i].set_title(f'{metric} vs Parameter Count', fontsize=14, fontweight='bold')
        axes[i].set_xlabel('Parameter Count (Millions)', fontsize=12)
        axes[i].set_ylabel(metric, fontsize=12)
        axes[i].grid(True, alpha=0.3)
        axes[i].legend(fontsize=10)
    
    plt.tight_layout()
    
    if save_figures:
        analysis_dir = os.path.join(results_dir, "model_analysis")
        os.makedirs(analysis_dir, exist_ok=True)
        
        png_path = os.path.join(analysis_dir, "parameter_vs_performance.png")
        eps_path = os.path.join(analysis_dir, "parameter_vs_performance.eps")
        
        plt.savefig(png_path, dpi=300, bbox_inches='tight')  # FIXED: dpi not dio
        plt.savefig(eps_path, format='eps', bbox_inches='tight')
        print(f"✓ Saved parameter vs performance analysis to: {png_path}")
    
    plt.show()
    plt.close()

def plot_comprehensive_training_analysis(results_df, training_histories, save_figures=True):
    """Plot comprehensive training analysis across all models"""
    neural_models = list(results_df['Model'].unique())
    
    if not training_histories:
        print("No training histories available for comprehensive analysis")
        return
    
    # Plot final training losses comparison
    plt.figure(figsize=(12, 8))
    
    final_train_losses = []
    final_val_losses = []
    model_names = []
    
    for model in neural_models:
        model_histories = [hist for key, hist in training_histories.items() if model in key]
        if model_histories:
            train_losses, val_losses = model_histories[-1]
            if train_losses:
                final_train_losses.append(train_losses[-1])
                final_val_losses.append(val_losses[-1] if val_losses else train_losses[-1])
                model_names.append(model)
    
    if final_train_losses:
        x_pos = np.arange(len(model_names))
        width = 0.35
        
        plt.bar(x_pos - width/2, final_train_losses, width, label='Final Train Loss', alpha=0.8)
        plt.bar(x_pos + width/2, final_val_losses, width, label='Final Val Loss', alpha=0.8)
        
        plt.xlabel('Models', fontsize=14)
        plt.ylabel('Final MSE Loss', fontsize=14)
        plt.title('Final Training vs Validation Loss Comparison', fontsize=16, fontweight='bold')
        plt.xticks(x_pos, model_names, rotation=45)
        plt.legend()
        plt.grid(True, alpha=0.3, axis='y')
        plt.yscale('log')
        
        if save_figures:
            analysis_dir = os.path.join(results_dir, "model_analysis")
            os.makedirs(analysis_dir, exist_ok=True)
            
            png_path = os.path.join(analysis_dir, "final_loss_comparison.png")
            plt.savefig(png_path, dpi=300, bbox_inches='tight')
            print(f"✓ Saved final loss comparison to: {png_path}")
        
        plt.show()
        plt.close()

# =============================
# 🔹


def plot_metrics_comparison(results_df, save_figures=True):
    models = results_df['Model'].unique()
    metrics = ['SNR', 'Correlation', 'PSNR', 'SSIM', 'MSE', 'MAE']

    fig, axes = plt.subplots(2, 3, figsize=(20, 12))
    axes = axes.flatten()

    for i, metric in enumerate(metrics):
        for model in models:
            model_data = results_df[results_df['Model'] == model]
            if metric in ['MSE', 'MAE']:
                axes[i].loglog(model_data['CR'], model_data[metric], 'o-', label=model, markersize=8, linewidth=2)
            else:
                axes[i].semilogx(model_data['CR'], model_data[metric], 'o-', label=model, markersize=8, linewidth=2)

        axes[i].set_title(f'{metric} vs Compression Ratio', fontsize=14, fontweight='bold')
        axes[i].set_xlabel('Compression Ratio', fontsize=12)
        axes[i].set_ylabel(metric, fontsize=12)
        axes[i].grid(True, alpha=0.3)
        axes[i].legend(fontsize=10)

    plt.tight_layout()

    if save_figures:
        png_path = os.path.join(results_dir, "comprehensive_metrics_comparison.png")
        eps_path = os.path.join(results_dir, "comprehensive_metrics_comparison.eps")

        plt.savefig(png_path, dpi=300, bbox_inches='tight')
        plt.savefig(eps_path, format='eps', bbox_inches='tight')
        print(f"\N{HEAVY CHECK MARK} Saved comprehensive metrics comparison to: {png_path}")

    plt.show()
    plt.close()


def plot_model_comparison(results_df, metric, save_figures=True):
    """Plot comparison of a specific metric across all models"""
    models = results_df['Model'].unique()
    cr_values = sorted(results_df['CR'].unique())

    plt.figure(figsize=(12, 8))

    for model in models:
        model_metrics = []
        for cr in cr_values:
            model_cr_data = results_df[(results_df['Model'] == model) & (results_df['CR'] == cr)]
            if not model_cr_data.empty:
                model_metrics.append(model_cr_data[metric].values[0])
            else:
                model_metrics.append(np.nan)

        if metric in ['MSE', 'MAE']:
            plt.loglog(cr_values, model_metrics, 'o-', label=model, markersize=8, linewidth=2)
        else:
            plt.semilogx(cr_values, model_metrics, 'o-', label=model, markersize=8, linewidth=2)

    plt.title(f'{metric} Comparison Across Models', fontsize=16, fontweight='bold')
    plt.xlabel('Compression Ratio', fontsize=14)
    plt.ylabel(metric, fontsize=14)
    plt.grid(True, alpha=0.3)
    plt.legend(fontsize=12)

    if save_figures:
        metric_dir = os.path.join(results_dir, "metric_comparisons")
        os.makedirs(metric_dir, exist_ok=True)

        png_path = os.path.join(metric_dir, f"{metric}_comparison.png")
        eps_path = os.path.join(metric_dir, f"{metric}_comparison.eps")

        plt.savefig(png_path, dpi=300, bbox_inches='tight')
        plt.savefig(eps_path, format='eps', bbox_inches='tight')
        print(f"\N{HEAVY CHECK MARK} Saved {metric} comparison to: {png_path}")

    plt.show()
    plt.close()


def plot_individual_model_metrics(results_df, save_figures=True):
    """Plot metrics for each individual model"""
    models = results_df['Model'].unique()
    metrics = ['SNR', 'PSNR', 'SSIM', 'MSE']

    for model in models:
        model_data = results_df[results_df['Model'] == model]

        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        axes = axes.flatten()

        for i, metric in enumerate(metrics):
            if metric in ['MSE']:
                axes[i].loglog(model_data['CR'], model_data[metric], 'bo-', markersize=8, linewidth=2)
            else:
                axes[i].semilogx(model_data['CR'], model_data[metric], 'bo-', markersize=8, linewidth=2)

            axes[i].set_title(f'{model} - {metric} vs CR', fontsize=14, fontweight='bold')
            axes[i].set_xlabel('Compression Ratio', fontsize=12)
            axes[i].set_ylabel(metric, fontsize=12)
            axes[i].grid(True, alpha=0.3)

        plt.tight_layout()

        if save_figures:
            model_metrics_dir = os.path.join(results_dir, model, "metrics")
            os.makedirs(model_metrics_dir, exist_ok=True)

            png_path = os.path.join(model_metrics_dir, f"{model}_metrics_summary.png")
            eps_path = os.path.join(model_metrics_dir, f"{model}_metrics_summary.eps")

            plt.savefig(png_path, dpi=300, bbox_inches='tight')
            plt.savefig(eps_path, format='eps', bbox_inches='tight')
            print(f"\N{HEAVY CHECK MARK} Saved {model} metrics summary to: {png_path}")

        plt.show()
        plt.close()


In [ ]:
# =============================
# Model comparison loop
# =============================
# (model_name, class/factory, extra constructor kwargs)
MODELS_TO_COMPARE = [
    (MAIN_MODEL_NAME, LARA, {"variant": LARA_VARIANT}),
    ("GeneralizedAutoencoder", GeneralizedAutoencoder, {}),
    ("AE_PureConcat", AE_PureConcat, {}),
    ("Wavelet_DB4", WaveletDB4, {}),
    ("Wavelet_SYM8", WaveletSYM8, {}),
    ("Wavelet_COIF3", WaveletCOIF3, {}),
]

results = []
training_histories = {}
run_t0 = time.time()

print(f"Starting model comparison over {len(MODELS_TO_COMPARE)} models x {len(CRS)} CRs")
print("=" * 80)

for model_name, model_class, model_kwargs in MODELS_TO_COMPARE:
    print(f"\n=== Processing {model_name} Model ===")

    for cr in CRS:
        print(f"\nProcessing {model_name} CR={cr}")
        print("-" * 40)

        try:
            if "Wavelet" in model_name:
                # ---- wavelet thresholding baseline: no training, evaluate directly ----
                model = model_class(cr)
                metrics = evaluate_model(model, test_data_np, cr, model_name)
                results.append(metrics)

                orig, recon = metrics["Reconstructions"]
                if SAVE_FIGS:
                    plot_waveforms_comprehensive(orig, recon, cr, model_name, save_figures=True)
                    plot_frequency_spectrum_all_models(orig, recon, cr, model_name, save_figures=True)

                print(f"{model_name} CR={cr}: SNR={metrics['SNR']:.2f}dB, "
                      f"Corr={metrics['Correlation']:.4f}, MSE={metrics['MSE']:.6f}, "
                      f"PSNR={metrics['PSNR']:.2f}dB, SSIM={metrics['SSIM']:.4f}")

            else:
                # ---- neural model: fresh checkpoint, train, then evaluate ----
                model_file = best_file(model_name, cr)
                if os.path.exists(model_file):
                    os.remove(model_file)

                if model_name == "AE_PureConcat" and cr not in model_2_architecture:
                    print(f"Skipping CR={cr} for AE_PureConcat (not in architecture config)")
                    continue

                model = model_class(cr, **model_kwargs)

                if model_name == MAIN_MODEL_NAME:
                    # LARA rebuilds itself through create_fallback_architecture() if the
                    # dimension assertion fails; variant is reset there, so this catches it.
                    assert getattr(model, "variant", None) == LARA_VARIANT, (
                        f"LARA fell back to the fallback architecture at CR={cr}")
                    trained_model, train_losses, val_losses = train_model_main(
                        model, train_loader, val_loader, model_name, cr, recipe=LARA_RECIPE)
                else:
                    trained_model, train_losses, val_losses = train_model_general(
                        model, train_loader, val_loader, model_name, cr)

                training_histories[f"{model_name}_CR{cr}"] = (train_losses, val_losses)

                if SAVE_FIGS:
                    plot_training_history(train_losses, val_losses, model_name, cr, save_figures=True)

                metrics = evaluate_model(trained_model, test_data, cr, model_name)
                results.append(metrics)

                orig, recon = metrics["Reconstructions"]
                if SAVE_FIGS:
                    plot_waveforms_comprehensive(orig, recon, cr, model_name, save_figures=True)
                    plot_frequency_spectrum_all_models(orig, recon, cr, model_name, save_figures=True)

                print(f"{model_name} CR={cr}: Actual CR={metrics['Actual_CR']:.2f}, "
                      f"SNR={metrics['SNR']:.2f}dB (med {metrics['SNR_median']:.2f}), "
                      f"Corr={metrics['Correlation']:.4f}, MSE={metrics['MSE']:.6f}, "
                      f"PSNR={metrics['PSNR']:.2f}dB, SSIM={metrics['SSIM']:.4f}, "
                      f"GradL1={metrics['GradL1']:.5f}, "
                      f"Params={metrics['Parameter_Count']:,}, Size={metrics['Model_Size_MB']:.2f}MB")

        except Exception as e:
            print(f"Error processing {model_name} CR={cr}: {e}")
            import traceback
            traceback.print_exc()
            continue

print(f"\nAll rows processed in {(time.time() - run_t0) / 60.0:.1f} min.")


In [ ]:
# =============================
# Results, figures, statistics
# =============================
RESULTS_CSV = os.path.join(results_dir, "model_comparison_results.csv")

if results:
    # Reconstructions / SNR_samples are large and not meaningful in a spreadsheet
    results_clean = [{k: v for k, v in m.items() if k not in ("Reconstructions", "SNR_samples")}
                     for m in results]
    results_df = pd.DataFrame(results_clean)
    results_df.to_csv(RESULTS_CSV, index=False)
    print(f"\N{HEAVY CHECK MARK} Results saved to: {RESULTS_CSV}")

    view_cols = ["Model", "CR", "Actual_CR", "SNR", "SNR_std", "SNR_median", "SNR_P25", "SNR_P75",
                 "SSIM", "SSIM_std", "SSIM_median", "PSNR", "GradL1", "Correlation",
                 "MSE", "Parameter_Count", "Model_Size_MB"]
    view_cols = [c for c in view_cols if c in results_df.columns]

    print("\nPer-CR results:")
    print("=" * 120)
    print(results_df[view_cols].round(4).to_string(index=False))

    # ----- aggregate figures -----
    try:
        plot_metrics_comparison(results_df, save_figures=SAVE_FIGS)
        for comp_metric in ["SNR", "PSNR", "SSIM", "MSE"]:
            plot_model_comparison(results_df, comp_metric, save_figures=SAVE_FIGS)
        plot_individual_model_metrics(results_df, save_figures=SAVE_FIGS)
        plot_parameter_comparison(results_df, save_figures=SAVE_FIGS)
        plot_parameter_vs_performance(results_df, save_figures=SAVE_FIGS)
        plot_comprehensive_training_analysis(results_df, training_histories, save_figures=SAVE_FIGS)
    except Exception as e:
        print(f"(aggregate figures skipped: {e})")

    # ----- CR-exactness check (neural models only; wavelets hit the CR by construction) -----
    neural = results_df[results_df['Model'].str.contains("Wavelet") == False]  # noqa: E712
    bad = [int(r["CR"]) for _, r in neural.iterrows()
           if abs(float(r["Actual_CR"]) - float(r["CR"])) > 0.5]
    print(f"\nCR-exactness (neural models): "
          f"{'OK - all rows match the requested CR' if not bad else 'FAIL at CRs ' + str(bad)}")

    # ----- best-weight ledger -----
    print("\nBest-weight ledger:")
    for model_name, _, _ in MODELS_TO_COMPARE:
        if "Wavelet" in model_name:
            continue
        present = sum(os.path.exists(best_file(model_name, cr)) for cr in CRS)
        print(f"  {model_name}: {present}/{len(CRS)} .pth files in {os.path.abspath(CKPT_DIR)}")

    # ----- per-model summary -----
    summary_rows = []
    for model_name in results_df['Model'].unique():
        md_ = results_df[results_df['Model'] == model_name]
        summary_rows.append({
            "Model": model_name,
            "CRs_run": len(md_),
            "Mean_SNR_dB": md_["SNR"].mean(),
            "Median_SNR_dB": md_["SNR_median"].mean(),
            "Mean_SSIM": md_["SSIM"].mean(),
            "Mean_GradL1": md_["GradL1"].mean(),
            "Mean_Params": md_["Parameter_Count"].mean(),
            "Mean_Size_MB": md_["Model_Size_MB"].mean(),
        })
    summary_df = pd.DataFrame(summary_rows)
    print("\nSummary (means over all CRs of each model):")
    print(summary_df.round(4).to_string(index=False))
    summary_csv = os.path.join(results_dir, "model_comparison_summary.csv")
    summary_df.to_csv(summary_csv, index=False)
    print(f"\N{HEAVY CHECK MARK} Summary saved to: {summary_csv}")

    # ----- model efficiency (neural models only) -----
    print("\nModel efficiency (mean SNR per million parameters):")
    print("-" * 60)
    for model_name in results_df['Model'].unique():
        if "Wavelet" in model_name:
            continue
        md_ = results_df[results_df['Model'] == model_name]
        avg_snr = md_["SNR"].mean()
        avg_params = md_["Parameter_Count"].mean() / 1e6
        if avg_params > 0:
            print(f"  {model_name:<24} SNR={avg_snr:6.2f}dB  params={avg_params:6.2f}M  "
                  f"size={md_['Model_Size_MB'].mean():6.2f}MB  "
                  f"efficiency={avg_snr / avg_params:7.2f} dB/M-params")
        else:
            print(f"  {model_name:<24} SNR={avg_snr:6.2f}dB  params=0")

    # ----- statistical significance -----
    print("\nStatistical significance testing (SNR across CRs):")
    p_values = statistical_significance_test(results_df, "SNR")
    for comparison, p_value in p_values.items():
        if not np.isfinite(p_value):
            print(f"  {comparison}: n/a (insufficient CRs)")
            continue
        significance = "***" if p_value < 0.001 else "**" if p_value < 0.01 else \
                       "*" if p_value < 0.05 else "ns"
        print(f"  {comparison}: p = {p_value:.4f} {significance}")

else:
    print("No results collected.")

print("\n" + "=" * 80)
print("Model comparison training completed!")
print(f"\N{HEAVY CHECK MARK} All results and figures saved in: {os.path.abspath(results_dir)}")
print(f"\N{HEAVY CHECK MARK} Best model weights (.pth) saved in: {os.path.abspath(CKPT_DIR)}")
print(f"\N{HEAVY CHECK MARK} Total elapsed: {(time.time() - run_t0) / 60.0:.1f} min")
print("=" * 80)
